# 1. Vom Random Forest zum neuronalen Netz — Betrugserkennung beim Kfz-Versicherer

**Session 6 · Dauer: 30–60 Min**

## Real-World-Kontext

In Kapitel 4 hast Du für genau dieses Problem — Betrugserkennung anhand von **Fahreralter**,
**Schadenshistorie** und **Fahrzeugtyp** — bereits ein KNN- und ein Random-Forest-Modell trainiert.
Beide Modelle liefen auf einer überschaubaren Tabelle mit ein paar hundert Fällen, und beide
funktionierten gut.

Heute lösen wir **exakt dieselbe Aufgabe noch einmal** — dieses Mal aber mit einem **neuronalen
Netz**. Das ist auch der Punkt, an dem wir ehrlich sein müssen, genau wie es auf der
Vorlesungsfolie "Zwei Beispiele aus dem Versicherer-Portfolio" steht:

> Für 500 Fälle mit 20 Spalten reicht ein Random Forest völlig aus — Deep Learning ist für diese
> Datenmenge **nicht sinnvoll**.

Wir bauen das neuronale Netz hier trotzdem — **nicht, weil es hier das beste Werkzeug wäre**,
sondern weil Du an einem Datensatz, den Du schon kennst, die komplette DL-Mechanik einmal von
Hand durchspielen sollst: Neuron → Schichten → Forward Pass → Loss → Training → PyTorch. Das
Verständnis, das Du hier aufbaust, brauchst Du in Kapitel 7 (Bilder) und Kapitel 8 (Text) wieder
— dort lohnt sich Deep Learning dann wirklich.

## 🎯 Learning Objectives

By completing this notebook, you will be able to:
- 💡 **Explain** wie ein künstliches Neuron aus gewichteter Summe und Aktivierungsfunktion eine
  Vorhersage berechnet, und den **Forward Pass** eines kleinen Netzes von Hand nachrechnen.
- 🔍 **Identify** die vier zentralen Aktivierungsfunktionen (Sigmoid, ReLU, Tanh, Softmax) anhand
  ihrer Kurvenform und wissen, wo sie typischerweise eingesetzt werden.
- 🛠️ **Apply** `PyTorch` (`nn.Module`, `nn.Linear`, `nn.ReLU`), um ein eigenes neuronales Netz
  (`VersichererMLP`) für ein tabellarisches Klassifikationsproblem zu definieren und zu trainieren.
- 🛠️ **Apply** den vollständigen Trainingsloop (Forward Pass → Loss → `zero_grad()` →
  `backward()` → `optimizer.step()`) über mehrere Epochen und Batches.
- 🔍 **Identify** anhand der Loss-Kurve und der Test-Accuracy, ob ein trainiertes Netz sinnvoll
  gelernt hat.
- ⚖️ **Bewerten**, wann ein Random-Forest-Modell (Kapitel 4) einem neuronalen Netz auf
  tabellarischen Daten in der Praxis vorzuziehen ist.

## Concept at a Glance

**Analogie — das Neuron als Lichtschalter:** Stell Dir ein künstliches Neuron als einen
**Lichtschalter mit Dimmer** vor. Der Schalter bekommt mehrere Signale gleichzeitig rein (z. B.
Fahreralter, Schadenshistorie, Fahrzeugtyp), gewichtet jedes Signal unterschiedlich stark (manche
Signale sind wichtiger als andere) und addiert alles zu einer einzigen Zahl auf. Diese Zahl
entscheidet dann — über eine **Aktivierungsfunktion** —, wie stark der "Dimmer" am Ende leuchtet:
schwach (kein Betrug) oder stark (Betrugsverdacht).

Mathematisch (aus der Vorlesung bekannt):

$$z = \sum_{i=1}^{m} w_i x_i + b \qquad a = \sigma(z)$$

- $x_i$ sind die Eingaben (unsere drei Merkmale)
- $w_i$ sind die **Gewichte** — wie stark jedes Merkmal zählt (trainierbar)
- $b$ ist der **Bias** — eine Grundverschiebung (trainierbar)
- $\sigma$ ist die **Aktivierungsfunktion**, die die Nichtlinearität reinbringt

> 💡 **Good to know:**
> Ein einzelnes Neuron ist im Kern nichts anderes als eine gewichtete Summe plus eine kleine
> "Verzerrung" der Ausgabe. Das Spannende an neuronalen Netzen entsteht erst, wenn man viele
> davon in **Schichten** (Input → Hidden → Output) hintereinanderschaltet.

Unser Running Example bleibt exakt dasselbe wie in Kapitel 4 und 5: ein Kfz-Versicherer mit drei
Merkmalen pro Kunde — **Fahreralter**, **Schadenshistorie** und **Fahrzeugtyp** — und der
Zielvariable **Betrug (Ja/Nein)**.

## Schritt 1 — Forward Pass von Hand nachrechnen

Bevor wir irgendetwas in PyTorch bauen, rechnen wir **ein einzelnes Neuron von Hand** nach — mit
genau dem Beispiel aus der Vorlesungsfolie "Der Forward Pass": Input `[25, 3, 1]`
(Fahreralter, Schadenshistorie, Fahrzeugtyp) ergibt am Ende $\sigma(0{,}68) = 0{,}66$, also eine
Betrugswahrscheinlichkeit von 66 %.

Wir wählen dafür feste Gewichte und einen Bias, die genau diesen gewichteten Summenwert $z=0{,}68$
ergeben, und aktivieren das Ergebnis anschließend mit der Sigmoid-Funktion.

In [ ]:
# I DO: Forward Pass eines einzelnen Neurons - Schritt für Schritt

import numpy as np

# Eingaben aus der Vorlesungsfolie: [Fahreralter, Schadenshistorie, Fahrzeugtyp]
x = np.array([25, 3, 1])

# Gewichte (frei gewaehlt, aber so, dass z = 0.68 herauskommt - siehe Vorlesungsfolie)
w = np.array([0.02, 0.1, 0.3])
b = -0.42

# Schritt 1: gewichtete Summe z = w . x + b (np.dot berechnet das Skalarprodukt)
z = np.dot(w, x) + b
print(f"Gewichtete Summe z = {z:.2f}")

Jetzt aktivieren wir $z$ mit der **Sigmoid-Funktion** — genau die, die Du aus der Vorlesung für
binäre Klassifikation kennst:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
# I DO: Aktivierung mit Sigmoid

def sigmoid(z):
    # 1 / (1 + e^(-z)) -> quetscht jeden Wert in den Bereich (0, 1)
    return 1 / (1 + np.exp(-z))

a = sigmoid(z)
print(f"Aktivierte Ausgabe a = sigmoid(z) = {a:.2f}")
print(f"-> Betrugswahrscheinlichkeit: {a * 100:.0f}%")

### Schritt-für-Schritt-Breakdown

1. `np.dot(w, x)` berechnet $w_1 x_1 + w_2 x_2 + w_3 x_3$ — die gewichtete Summe der drei
   Merkmale, exakt wie in der Formel $z = \sum_i w_i x_i + b$.
2. `+ b` addiert den **Bias** — eine Grundverschiebung, unabhängig von den Eingabewerten.
3. `sigmoid(z)` quetscht das Ergebnis in den Bereich (0, 1) — direkt als Wahrscheinlichkeit
   interpretierbar.

> 💡 **Good to know:**
> $z \approx 0{,}68$ und $\sigma(0{,}68) \approx 0{,}66$ — das deckt sich mit der
> Vorlesungsfolie "66 % Betrugswahrscheinlichkeit". Die konkreten Gewichte sind hier frei
> gewählt (sie werden im echten Training gelernt, nicht von Hand gesetzt) — nur das *Ergebnis*
> soll zur Folie passen.

Jetzt probierst Du selbst aus, was passiert, wenn sich die Gewichte ändern.

> 🎯 **Your Task:**
> Verändere in der Zelle unten die Gewichte `w_neu` (z. B. gib der Schadenshistorie ein größeres
> Gewicht) und beobachte, wie sich `z` und die aktivierte Ausgabe verändern.

In [ ]:
# WE DO: Andere Gewichte einsetzen und Effekt beobachten

w_neu = np.array([0.02, 0.25, 0.3])  # <- veraendere hier z.B. das mittlere Gewicht (Schadenshistorie)
b_neu = -0.42

z_neu = np.dot(w_neu, x) + b_neu
a_neu = sigmoid(z_neu)

print(f"Neue gewichtete Summe z = {z_neu:.2f}")
print(f"Neue aktivierte Ausgabe a = {a_neu:.2f} (vorher: {a:.2f})")

### Zwischenfazit

Du hast gerade den kompletten Forward Pass eines einzelnen Neurons nachgerechnet — genau die
Mechanik, die in einem echten Netz **tausendfach parallel** in jeder Schicht abläuft. Bevor wir
ein ganzes Netz bauen, schauen wir uns die Aktivierungsfunktionen noch etwas genauer an: Sigmoid
ist nicht die einzige Option.

## Schritt 2 — Aktivierungsfunktionen implementieren & vergleichen

Aus der Vorlesung kennst Du vier zentrale Aktivierungsfunktionen. Wir implementieren sie jetzt
alle selbst mit `numpy` und plotten sie nebeneinander, damit Du ihre charakteristische Form
direkt siehst.

- **Sigmoid** — $\sigma(z) = \dfrac{1}{1+e^{-z}}$, Bereich (0, 1), Standard für binäre Klassifikation im Output.
- **ReLU** — $\text{ReLU}(z) = \max(0, z)$, Standard in Hidden Layers.
- **Tanh** — $\tanh(z)$, wie Sigmoid, aber zentriert in (-1, 1).
- **Softmax** — wandelt einen ganzen Vektor in eine Wahrscheinlichkeitsverteilung um (Summe = 1), Standard für Multiklassen-Output.

In [ ]:
# I DO: Die vier Aktivierungsfunktionen als eigene Funktionen definieren

def relu(z):
    # Ist z positiv? Dann gib z durch. Sonst gib 0 zurueck.
    return np.maximum(0, z)

def tanh(z):
    # numpy bringt tanh schon fertig mit
    return np.tanh(z)

def softmax(z):
    # Softmax braucht einen ganzen Vektor, keinen Einzelwert - wir subtrahieren das Maximum
    # zuerst ab, das ist ein Standard-Trick fuer numerische Stabilitaet (vermeidet Overflow).
    z_stabil = z - np.max(z)
    exp_werte = np.exp(z_stabil)
    return exp_werte / np.sum(exp_werte)

# Kurzer Test: Softmax auf einem Beispielvektor
beispiel_werte = np.array([2.0, 1.0, 0.1])
print("Softmax-Beispiel:", softmax(beispiel_werte).round(3), "Summe:", softmax(beispiel_werte).sum())

In [ ]:
# I DO: Sigmoid, ReLU und Tanh auf einem Wertebereich plotten

import matplotlib.pyplot as plt

z_werte = np.linspace(-6, 6, 200)  # 200 Punkte zwischen -6 und 6

fig, achsen = plt.subplots(1, 3, figsize=(15, 4))

achsen[0].plot(z_werte, sigmoid(z_werte), color="tab:blue")
achsen[0].set_title("Sigmoid")
achsen[0].axhline(0, color="gray", lw=0.5)
achsen[0].axvline(0, color="gray", lw=0.5)

achsen[1].plot(z_werte, relu(z_werte), color="tab:orange")
achsen[1].set_title("ReLU")
achsen[1].axhline(0, color="gray", lw=0.5)
achsen[1].axvline(0, color="gray", lw=0.5)

achsen[2].plot(z_werte, tanh(z_werte), color="tab:green")
achsen[2].set_title("Tanh")
achsen[2].axhline(0, color="gray", lw=0.5)
achsen[2].axvline(0, color="gray", lw=0.5)

for achse in achsen:
    achse.set_xlabel("z")
    achse.set_ylabel("Aktivierung")

plt.tight_layout()
plt.show()

> ⚠️ **Common Pitfall:**
> Softmax lässt sich nicht wie die anderen drei einfach "punktweise" auf ein einzelnes $z$
> plotten — sie braucht immer einen **ganzen Vektor** von Roh-Ausgaben (z. B. eine pro Klasse)
> und verteilt die Wahrscheinlichkeit über alle Klassen gemeinsam. Deshalb fehlt sie im
> Plot oben und wird stattdessen als Balkendiagramm dargestellt.

In [ ]:
# I DO: Softmax auf einem Beispiel mit 3 "Klassen" als Balkendiagramm

klassen = ["Kein Betrug", "Betrugsverdacht", "Fehlerhafte Angabe"]
rohwerte = np.array([1.2, 3.5, 0.4])  # z.B. Roh-Ausgaben der letzten Schicht fuer 3 Klassen
wahrscheinlichkeiten = softmax(rohwerte)

plt.figure(figsize=(5, 4))
plt.bar(klassen, wahrscheinlichkeiten, color="tab:red")
plt.ylabel("Wahrscheinlichkeit")
plt.title("Softmax: Roh-Ausgaben -> Wahrscheinlichkeitsverteilung")
plt.xticks(rotation=15)
plt.show()

print("Summe aller Wahrscheinlichkeiten:", wahrscheinlichkeiten.sum().round(4))

**Wann welche Funktion?** (Zusammenfassung von der Vorlesungsfolie)

| Funktion | Typischer Einsatz | Vorteil | Nachteil |
|:---|:---|:---|:---|
| **Sigmoid** | Output (binäre Klassifikation) | Wahrscheinlichkeitsinterpretation | Vanishing Gradients |
| **ReLU** | Hidden Layers (STANDARD) | Effizient, löst Vanishing Gradients | Dead Neurons möglich |
| **Tanh** | Spezielle Netze (RNN) | Zentriert um 0 | Vanishing Gradients |
| **Softmax** | Output (Multiklassen) | Echte Wahrscheinlichkeitsverteilung | Nur für Output |

### Zwischenfazit

Du kennst jetzt Bauplan (Neuron) und Werkzeugkasten (Aktivierungsfunktionen) für ein neuronales
Netz. Als Nächstes brauchen wir echte Daten, um so ein Netz überhaupt trainieren zu können.

## Schritt 3 — Synthetischer Versicherer-Datensatz

Wir bauen uns — analog zu Kapitel 4 und 5 — einen synthetischen, aber realistischen
Kfz-Versicherer-Datensatz mit denselben drei Merkmalen wie auf der Vorlesungsfolie:
**Fahreralter**, **Schadenshistorie** (Anzahl gemeldeter Schäden) und **Fahrzeugtyp**
(0 = Standard-Fahrzeug, 1 = Sportwagen). Die Zielvariable ist **Betrug** (0 = Nein, 1 = Ja).

> 💡 **Good to know:**
> Betrug ist in der Realität ein **seltenes Ereignis** — die allermeisten Versicherungsfälle
> sind legitim. Wir bilden das bewusst mit einer **Klassenimbalance** nach: Nur ein kleiner Teil
> unserer 350 simulierten Fälle ist tatsächlich Betrug.

In [ ]:
# I DO: Merkmale simulieren

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # fester Seed -> reproduzierbare Ergebnisse
n_kunden = 350

fahreralter = rng.uniform(18, 80, n_kunden)
schadenshistorie = rng.poisson(1.0, n_kunden).clip(0, 6)   # Anzahl gemeldeter Schaeden
fahrzeugtyp = rng.integers(0, 2, n_kunden)                  # 0 = Standard, 1 = Sportwagen

print(f"Anzahl Kunden: {n_kunden}")
print(f"Fahreralter: {fahreralter.min():.0f} - {fahreralter.max():.0f} Jahre")
print(f"Schadenshistorie: {schadenshistorie.min()} - {schadenshistorie.max()} Schaeden")

Jetzt konstruieren wir die Zielvariable `betrug`: junge Fahrer, viele vergangene Schäden und
Sportwagen erhöhen die Betrugswahrscheinlichkeit — insgesamt bleibt Betrug aber ein seltenes
Ereignis (realistische Klassenimbalance).

In [ ]:
# I DO: Zielvariable 'betrug' auf Basis der Merkmale simulieren (mit realistischem Rauschen)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Basis-Risiko-Score: negativer Achsenabschnitt haelt Betrug insgesamt selten
logit = (
    -3.3
    - 0.045 * (fahreralter - 25)   # aeltere Fahrer -> geringeres Risiko
    + 0.55 * schadenshistorie       # mehr Schaeden in der Vergangenheit -> hoeheres Risiko
    + 0.9 * fahrzeugtyp             # Sportwagen -> hoeheres Risiko
)
rauschen = rng.normal(0, 0.4, n_kunden)  # echtes Leben ist nie perfekt vorhersagbar
betrugswahrscheinlichkeit = sigmoid(logit + rauschen)
betrug = rng.binomial(1, betrugswahrscheinlichkeit)  # Muenzwurf mit dieser Wahrscheinlichkeit

kunden = pd.DataFrame({
    "fahreralter": fahreralter,
    "schadenshistorie": schadenshistorie,
    "fahrzeugtyp": fahrzeugtyp,
    "betrug": betrug,
})

print(f"Betrugsfaelle: {kunden['betrug'].sum()} von {len(kunden)} ({kunden['betrug'].mean() * 100:.1f}%)")
kunden.head()

> ⚠️ **Common Pitfall:**
> Mit nur ~6 % positiven Fällen ist unser Datensatz **stark unbalanciert**. Achte später bei der
> Bewertung darauf, dass eine hohe Accuracy allein wenig aussagt — ein Modell, das immer "kein
> Betrug" vorhersagt, hätte hier auch schon ~94 % Accuracy, obwohl es nutzlos ist!

## Train/Test-Split & Standardisierung

Genau wie in Kapitel 5 (und schon in Kapitel 4) gilt: **Skalierung vor dem Training ist Pflicht.**
Neuronale Netze reagieren besonders empfindlich auf unterschiedliche Wertebereiche — Gradient
Descent konvergiert auf unskalierten Daten deutlich langsamer und instabiler.

In [ ]:
# I DO: Train/Test-Split

from sklearn.model_selection import train_test_split

merkmale = ["fahreralter", "schadenshistorie", "fahrzeugtyp"]
X = kunden[merkmale].values
y = kunden["betrug"].values

# stratify=y sorgt dafuer, dass Trainings- und Testset denselben Betrugsanteil haben
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Trainingsfaelle: {len(X_train)} (davon Betrug: {y_train.sum()})")
print(f"Testfaelle:      {len(X_test)} (davon Betrug: {y_test.sum()})")

In [ ]:
# I DO: Standardisierung mit StandardScaler (nur auf den Trainingsdaten fitten!)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform auf Trainingsdaten
X_test_scaled = scaler.transform(X_test)          # nur transform auf Testdaten!

print("Mittelwert Trainingsdaten (sollte ~0 sein):", X_train_scaled.mean(axis=0).round(2))
print("Std Trainingsdaten (sollte 1 sein):        ", X_train_scaled.std(axis=0).round(2))

> 💡 **Good to know:**
> Warum skalieren wir auch hier wieder? Ohne Skalierung würde `fahreralter` (Werte bis 80) die
> Gewichtsupdates dominieren, während `fahrzeugtyp` (nur 0 oder 1) kaum Einfluss hätte — dasselbe
> Problem wie beim Clustering in Kapitel 5, nur diesmal wirkt es sich auf den **Gradient
> Descent** aus statt auf die Distanzberechnung. Und wie schon dort gilt: **fit** nur auf den
> Trainingsdaten, **transform** auf beiden — sonst "leaken" Informationen aus dem Testset ins
> Training.

### Zwischenfazit

Wir haben jetzt genau das, was ein Netz zum Training braucht: skalierte Features (`X_train_scaled`,
`X_test_scaled`) und die passenden Zielwerte (`y_train`, `y_test`). Zeit, das eigentliche Netz zu
bauen.

## Schritt 4 — `VersichererMLP` in PyTorch definieren

Jetzt übersetzen wir die Netz-Architektur aus der Vorlesungsfolie "Konkret: Unser Versicherer-MLP"
1:1 in PyTorch-Code. Die Architektur:

- **Input-Schicht:** 3 Merkmale (Fahreralter, Schadenshistorie, Fahrzeugtyp)
- **Hidden-Schicht:** 5 Neuronen, aktiviert mit ReLU
- **Output-Schicht:** 1 Neuron (Roh-Ausgabe, aus der wir später die Betrugswahrscheinlichkeit ableiten)

> 💡 **Good to know:**
> `nn` steht für **n**eural **n**etwork — `torch.nn` ist PyTorchs Modul für alles rund um
> neuronale Netze (Schichten, Aktivierungsfunktionen, Loss-Funktionen).

In [ ]:
# I DO: VersichererMLP als eigene Klasse definieren

import torch
import torch.nn as nn

torch.manual_seed(42)  # fester Seed -> reproduzierbare Gewichts-Initialisierung

class VersichererMLP(nn.Module):
    def __init__(self):
        # super().__init__() ruft zuerst PyTorchs eigene Initialisierung auf - Pflicht bei
        # jeder Klasse, die von nn.Module erbt.
        super(VersichererMLP, self).__init__()
        self.fc1 = nn.Linear(3, 5)   # Input (3 Merkmale) -> Hidden (5 Neuronen)
        self.relu = nn.ReLU()        # Aktivierungsfunktion zwischen den Schichten
        self.fc2 = nn.Linear(5, 1)   # Hidden (5 Neuronen) -> Output (1 Vorhersage)

    def forward(self, x):
        # forward() beschreibt, wie die Daten durch das Netz fliessen (der Forward Pass!)
        x = self.fc1(x)   # gewichtete Summe der Input-Schicht (inkl. Bias, automatisch von PyTorch verwaltet)
        x = self.relu(x)  # Nichtlinearitaet reinbringen
        return self.fc2(x)  # gewichtete Summe der Output-Schicht -> Roh-Ausgabe (noch kein Sigmoid!)

model = VersichererMLP()
print(model)

> ⚠️ **Common Pitfall:**
> Die `forward()`-Methode gibt hier **keine** Sigmoid-aktivierte Wahrscheinlichkeit zurück,
> sondern die Roh-Ausgabe (den sogenannten **Logit**)! Das ist Absicht — mehr dazu im nächsten
> Schritt bei der Wahl der Loss-Funktion.

`nn.Linear(3, 5)` speichert intern genau die Gewichte $w$ und den Bias $b$ aus der
Neuron-Formel von Schritt 1 — für alle 5 Neuronen der Hidden-Schicht gleichzeitig. PyTorch
initialisiert diese Werte automatisch mit kleinen Zufallszahlen (nicht 0 — sonst würden alle
Neuronen identisch lernen).

## Schritt 5 — Loss-Funktion & Optimizer wählen

Die Vorlesungsfolie erwähnt zwei Loss-Funktionen: `MSELoss` für Regression (z. B. Schadenshöhe
vorhersagen) und `CrossEntropyLoss` für Klassifikation. Unser Betrug-Ja/Nein-Problem ist eine
**binäre Klassifikation** — dafür nutzen wir aber genauer `nn.BCEWithLogitsLoss()` statt
`nn.CrossEntropyLoss()`.

> 💡 **Good to know:**
> `BCEWithLogitsLoss` ("Binary Cross-Entropy with Logits") ist mathematisch eine Cross-Entropy,
> aber speziell für **genau eine** Output-Wahrscheinlichkeit gebaut (statt einer Verteilung über
> mehrere Klassen wie bei `CrossEntropyLoss`, das für MNIST & Co. mit 10 Klassen gedacht ist). Der
> Zusatz "WithLogits" bedeutet: Die Funktion wendet **intern selbst** eine Sigmoid-Aktivierung auf
> die Roh-Ausgabe unseres Netzes an, bevor sie den Fehler berechnet — deshalb gibt `forward()`
> oben bewusst *keine* fertige Wahrscheinlichkeit zurück. Das ist numerisch stabiler, als Sigmoid
> und Cross-Entropy manuell getrennt zu rechnen.
> `MSELoss` bleibt für **Regressionsziele** reserviert (kontinuierliche Werte wie eine
> Schadenshöhe in Euro) — für unser Ja/Nein-Problem wäre MSE ein schlechterer Fehlermaßstab.

Als Optimizer nutzen wir — wie auf der Folie empfohlen — **Adam** mit `lr=0.001`.

In [ ]:
# WE DO: Loss-Funktion und Optimizer instanziieren

import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()                       # Fehlermass fuer binaere Klassifikation
optimizer = optim.Adam(model.parameters(), lr=0.001)      # Adam, wie auf der Vorlesungsfolie empfohlen

print("Loss-Funktion:", criterion)
print("Optimizer:", optimizer)

> 🎯 **Your Task:**
> Probiere in der Zelle unten eine **höhere Lernrate** (z. B. `lr=0.05`) aus und trainiere ein
> paar Testschritte (nur zur Beobachtung — der eigentliche Trainingsloop kommt erst in Schritt 6).
> Beobachte, wie stark sich der Loss von Schritt zu Schritt verändert.

In [ ]:
# WE DO: Guided Tweak - Effekt einer anderen Lernrate auf ein paar Testschritte beobachten

model_test = VersichererMLP()
torch.manual_seed(42)
optimizer_test = optim.Adam(model_test.parameters(), lr=0.05)  # <- hier experimentieren (z.B. 0.05 oder 0.0001)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

for schritt in range(5):
    output = model_test(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    optimizer_test.zero_grad()
    loss.backward()
    optimizer_test.step()
    print(f"Schritt {schritt + 1}: Loss = {loss.item():.4f}")

> ⚠️ **Common Pitfall:**
> Eine zu hohe Lernrate (z. B. `lr=0.05` oder größer) lässt den Loss oft **instabil springen**
> statt gleichmäßig zu sinken — genau das "Überschießen", das auf der Vorlesungsfolie zur
> Lernrate gezeigt wurde. `lr=0.001` ist für unser Netz die deutlich stabilere Wahl.

### Zwischenfazit

Modell, Loss-Funktion und Optimizer stehen. Jetzt bauen wir daraus den vollständigen
Trainingsloop — mit echten Batches über mehrere Epochen.

## Schritt 6 — Der vollständige Trainingsloop

Jetzt kommt das Muster aus der Vorlesungsfolie "Training in Code": Ein `DataLoader` liefert uns
automatisch einen Batch nach dem anderen, und für jeden Batch durchlaufen wir dieselben fünf
Schritte:

1. **Forward Pass:** `output = model(batch_x)`
2. **Loss berechnen:** `loss = criterion(output, batch_y)`
3. **Gradienten zurücksetzen:** `optimizer.zero_grad()`
4. **Backward Pass:** `loss.backward()`
5. **Gewichte anpassen:** `optimizer.step()`

Das Ganze wiederholt sich über mehrere **Epochen** (vollständige Durchläufe durch alle
Trainingsdaten).

> 🎯 **Your Task:**
> Vervollständige die `# TODO`-Zeilen im Trainingsloop unten. Orientiere Dich am Muster aus der
> Vorlesungsfolie "End-to-End (2/2)".

In [ ]:
# I DO: Trainingsdaten in ein PyTorch-Dataset + DataLoader packen

from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # Form (n, 1) statt (n,)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # 32 Faelle pro Batch

print(f"Anzahl Batches pro Epoch: {len(train_loader)}")

In [ ]:
# =========================================================================
# 🎯 EXERCISE: Trainingsloop vervollstaendigen
# =========================================================================
# Instruction: Ersetze die TODO-Kommentare durch gueltigen Code (siehe Anleitung oben).

# Modell, Loss und Optimizer frisch initialisieren (reproduzierbar dank Seed)
torch.manual_seed(42)
model = VersichererMLP()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

anzahl_epochen = 60
loss_pro_epoche = []  # hier sammeln wir den durchschnittlichen Loss jeder Epoche

for epoch in range(anzahl_epochen):
    epoch_losses = []
    for batch_x, batch_y in train_loader:
        # 1. Forward Pass: Vorhersage des Modells fuer diesen Batch berechnen
        output = ???

        # 2. Loss berechnen: Fehler zwischen Vorhersage 'output' und echtem Label 'batch_y'
        loss = ???

        # 3. Gradienten aus dem letzten Schritt zuruecksetzen
        ???

        # 4. Backward Pass: Gradienten fuer alle Gewichte berechnen
        ???

        # 5. Gewichte anhand der Gradienten anpassen
        ???

        epoch_losses.append(loss.item())

    loss_pro_epoche.append(np.mean(epoch_losses))

    if (epoch + 1) % 10 == 0:
        print(f"Epoche {epoch + 1}/{anzahl_epochen}: Loss = {loss_pro_epoche[-1]:.4f}")

print("Training abgeschlossen.")

> 💡 **Good to know:**
> `optimizer.zero_grad()` **muss** vor jedem `loss.backward()` aufgerufen werden — sonst würden
> sich die Gradienten mit denen aus dem vorherigen Batch **aufaddieren**, statt für den aktuellen
> Batch neu berechnet zu werden. Das ist einer der häufigsten Anfängerfehler in PyTorch!

## Schritt 7 — Evaluation: Hat das Netz etwas gelernt?

Zunächst schauen wir uns die **Loss-Kurve** über alle Epochen an — sie sollte, wie auf der
Vorlesungsfolie zur Trainingsmechanik gezeigt, mit hohem Loss starten und dann abfallen.

In [ ]:
# I DO: Loss-Kurve ueber alle Epochen plotten

plt.figure(figsize=(7, 5))
plt.plot(range(1, anzahl_epochen + 1), loss_pro_epoche, color="tab:purple")
plt.xlabel("Epoche")
plt.ylabel("Durchschnittlicher Loss (BCEWithLogitsLoss)")
plt.title("Trainingsverlauf: Loss pro Epoche")
plt.show()

Jetzt prüfen wir, wie gut das Netz auf **nie gesehenen** Testdaten performt. Dafür wandeln wir
die Roh-Ausgaben (Logits) mit Sigmoid in Wahrscheinlichkeiten um und schneiden bei 0,5 in
"Betrug" / "kein Betrug".

> ⚠️ **Common Pitfall:**
> `model.eval()` und `torch.no_grad()` sind beim Evaluieren wichtig: `eval()` schaltet
> Trainings-spezifisches Verhalten aus (bei unserem einfachen Netz macht das noch keinen
> Unterschied, ist aber gute Praxis), und `no_grad()` sagt PyTorch, dass es hier **keine**
> Gradienten berechnen muss — das spart Rechenzeit und Speicher bei reinen Vorhersagen.

In [ ]:
# I DO: Vorhersagen auf dem Testset berechnen

model.eval()  # Modell in den Evaluierungsmodus schalten

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

with torch.no_grad():  # keine Gradienten noetig, wir wollen nur vorhersagen
    logits_test = model(X_test_tensor)
    wahrscheinlichkeiten_test = torch.sigmoid(logits_test)  # Logits -> Wahrscheinlichkeiten
    vorhersagen_test = (wahrscheinlichkeiten_test >= 0.5).int().squeeze().numpy()

genauigkeit = (vorhersagen_test == y_test).mean()
print(f"Test-Accuracy: {genauigkeit * 100:.1f}%")

In [ ]:
# I DO: Confusion Matrix (Anschluss an Kapitel 4)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

matrix = confusion_matrix(y_test, vorhersagen_test)
anzeige = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=["Kein Betrug", "Betrug"])
anzeige.plot(cmap="Blues")
plt.title("Confusion Matrix - VersichererMLP auf Testdaten")
plt.show()

> 💡 **Good to know:**
> Wegen der starken Klassenimbalance (nur ~6 % Betrug) lohnt sich ein zweiter Blick auf die
> Confusion Matrix statt nur auf die Accuracy: Wie viele der **tatsächlichen** Betrugsfälle
> (untere Zeile) hat das Netz überhaupt erkannt? Das ist die eigentlich interessante Zahl bei
> einem seltenen Ereignis.

### Zwischenfazit

Du hast jetzt ein komplettes neuronales Netz trainiert und evaluiert — von der Architektur über
den Trainingsloop bis zur Auswertung. Zeit für die abschließende Reflexion.

## Mini-Exercise / Self-Check

**Teil A — Hyperparameter experimentell verändern:**

> 🎯 **Your Task:**
> Verändere in der Zelle unten **eine** der beiden Stellschrauben — entweder die Größe der
> Hidden-Schicht (z. B. `nn.Linear(3, 20)` statt `nn.Linear(3, 5)`, dann natürlich auch
> `nn.Linear(20, 1)` für `fc2`) oder die Lernrate — trainiere erneut und beobachte den Effekt auf
> Loss-Kurve und Test-Accuracy. Trage Deine Beobachtung anschließend als Kommentar ein.

In [ ]:
# =========================================================================
# 🎯 EXERCISE: Eigenes Experiment - Hidden-Layer-Groesse oder Lernrate veraendern
# =========================================================================
# Instruction: Aendere GENAU EINEN Hyperparameter (Hidden-Groesse ODER Lernrate),
# trainiere neu und interpretiere den Effekt in der Kommentarzeile am Ende.

class VersichererMLPExperiment(nn.Module):
    def __init__(self):
        super(VersichererMLPExperiment, self).__init__()
        self.fc1 = nn.Linear(3, ???)   # z.B. 20 statt 5 Neuronen
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(???, 1)   # muss zur Hidden-Groesse oben passen!

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        return self.fc2(x)

torch.manual_seed(42)
model_exp = VersichererMLPExperiment()
optimizer_exp = optim.Adam(model_exp.parameters(), lr=???)  # z.B. 0.001, 0.01 oder 0.0001

loss_pro_epoche_exp = []
for epoch in range(anzahl_epochen):
    epoch_losses = []
    for batch_x, batch_y in train_loader:
        output = model_exp(batch_x)
        loss = criterion(output, batch_y)
        optimizer_exp.zero_grad()
        loss.backward()
        optimizer_exp.step()
        epoch_losses.append(loss.item())
    loss_pro_epoche_exp.append(np.mean(epoch_losses))

model_exp.eval()
with torch.no_grad():
    logits_exp = model_exp(X_test_tensor)
    vorhersagen_exp = (torch.sigmoid(logits_exp) >= 0.5).int().squeeze().numpy()
genauigkeit_exp = (vorhersagen_exp == y_test).mean()

print(f"Finaler Trainings-Loss: {loss_pro_epoche_exp[-1]:.4f} (vorher: {loss_pro_epoche[-1]:.4f})")
print(f"Test-Accuracy: {genauigkeit_exp * 100:.1f}% (vorher: {genauigkeit * 100:.1f}%)")

# TODO: Trage hier Deine Beobachtung als Kommentar ein.
# Wurde das Training schneller/langsamer? Ist die Accuracy besser oder schlechter geworden?

**Teil B — Konzeptuelle Reflexion:**

> 🎯 **Your Task:**
> Warum wäre für diesen kleinen Datensatz in der Praxis eher **Random Forest aus Kapitel 4** die
> bessere Wahl als dieses neuronale Netz? Nenne **2 Gründe**.

In [ ]:
# TODO: Trage Deine zwei Gruende als Kommentare bzw. print()-Ausgaben ein.

grund_1 = "???"
grund_2 = "???"

print(f"Grund 1: {grund_1}")
print(f"Grund 2: {grund_2}")

## Summary & Key Takeaways

- Ein **künstliches Neuron** berechnet eine gewichtete Summe ($z = \sum_i w_i x_i + b$) und
  wendet dann eine **Aktivierungsfunktion** an ($a = \sigma(z)$) — genau das haben wir am
  Forward-Pass-Beispiel `[25, 3, 1] -> 0,66` von Hand nachgerechnet.
- **Aktivierungsfunktionen** bringen Nichtlinearität ins Netz: **Sigmoid** für binäre
  Klassifikation im Output, **ReLU** als Standard in Hidden Layers, **Tanh** als zentrierte
  Alternative, **Softmax** für Multiklassen-Output.
- Genau wie beim klassischen ML (Kapitel 4/5) gilt: **Skalierung ist Pflicht** — `StandardScaler`
  auf den Trainingsdaten fitten, auf beiden Sets transformieren.
- **`nn.Module`** ist die PyTorch-Klasse für neuronale Netze: `__init__` definiert die
  Architektur (`nn.Linear`, `nn.ReLU`), `forward()` definiert den Datenfluss.
- Der **Trainingsloop** folgt immer demselben Muster: Forward Pass → Loss →
  `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()` — pro Batch, über mehrere
  Epochen.
- **`BCEWithLogitsLoss`** ist die richtige Wahl für binäre Klassifikation mit einer
  Output-Wahrscheinlichkeit (statt `CrossEntropyLoss`, das für Multiklassen-Probleme gedacht
  ist) — sie kombiniert Sigmoid und Cross-Entropy intern und numerisch stabil.
- **Ehrliches Fazit:** Für diesen kleinen, tabellarischen Datensatz ist Random Forest aus
  Kapitel 4 in der Praxis die bessere Wahl (weniger Daten nötig, interpretierbar). Neuronale
  Netze zeigen ihre Stärke erst bei **unstrukturierten** Daten und großen Datenmengen — genau
  dort setzen die nächsten Kapitel an.

**Weiter geht's:** Im nächsten Kapitel verlassen wir Tabellendaten endgültig — dort geht es um
**Computer Vision** und Convolutional Neural Networks (CNNs) für Kfz-Schadensfotos.